# Astro Tabular NN: Quick CUDA Scout

This notebook is the first stage of a new research direction:

- **astro-only tabular features**
- **CUDA-required neural training**
- **Numba-accelerated margin scan**
- short scout runs to estimate potential and tuning ranges


In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "RESEARCH").exists():
    # notebook can be opened from its own folder
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


In [ ]:
from RESEARCH.astro_tabular_nn.best_grid_dataset import ensure_best_grid_dataset_path
from RESEARCH.astro_tabular_nn.config import (
    DatasetConfig,
    ScoutConfig,
    TrainConfig,
    with_dataset,
    with_epochs,
    with_batch_size,
)
from RESEARCH.astro_tabular_nn.data_utils import load_tabular_dataset, build_time_split, split_summary
from RESEARCH.astro_tabular_nn.experiments import (
    default_scout_model_grid,
    run_quick_scout,
    suggest_tuning_bounds,
)
from RESEARCH.astro_tabular_nn.postrun_report import render_postrun_report


In [ ]:
RUN_TAG = "turning_massive_label_grid"
DATA_START = "2017-11-01"
SCOUT_EPOCHS = 3
SCOUT_BATCH_SIZE = 512
SCOUT_SEEDS = (42,)

DATASET_PATH = ensure_best_grid_dataset_path(
    run_tag=RUN_TAG,
    data_start=DATA_START,
    use_cache=True,
    verbose=True,
)

base_ds_cfg = DatasetConfig()
ds_cfg = with_dataset(base_ds_cfg, DATASET_PATH)

base_train_cfg = TrainConfig()
train_cfg = with_epochs(base_train_cfg, SCOUT_EPOCHS)
train_cfg = with_batch_size(train_cfg, SCOUT_BATCH_SIZE)

scout_cfg = ScoutConfig(train=train_cfg)

ds_cfg, scout_cfg


In [ ]:
dataset = load_tabular_dataset(ds_cfg)
split = build_time_split(len(dataset.y), scout_cfg.split)
summary = split_summary(dataset, split)

pd.Series(summary)


In [ ]:
# Start with one strong spec per architecture, then expand if needed.
all_specs = default_scout_model_grid()
model_specs = [
    all_specs[0],  # dcn_base
    all_specs[3],  # dcn_deep_cross
    all_specs[4],  # deepfm_base
    all_specs[7],  # deepfm_embed64
]

for spec in model_specs:
    print(spec)


In [ ]:
results, meta = run_quick_scout(
    dataset=dataset,
    scout_cfg=scout_cfg,
    model_specs=model_specs,
    seeds=SCOUT_SEEDS,
    verbose=True,
)

postrun_metrics = render_postrun_report(
    results=results,
    run_rank=1,
    title="Quick Scout - Default Post-Run Diagnostics",
)

results.head(20)


In [ ]:
display_cols = [
    "model", "model_type", "seed", "cutoff_kind", "best_epoch", "best_margin", "best_val_score",
    "test_recall_down", "test_recall_up", "test_recall_min", "test_recall_gap", "test_mcc", "test_acc",
    "test_true_up_share", "test_pred_up_share", "test_true_balance_gap_ud", "test_pred_balance_gap_ud",
    "val_recall_down", "val_recall_up", "val_recall_min", "val_recall_gap", "val_mcc", "val_acc",
]
results[display_cols].sort_values(["test_recall_min", "test_mcc"], ascending=False)


In [ ]:
bounds = suggest_tuning_bounds(results, top_k=scout_cfg.top_k_for_bounds)
pd.Series(bounds)


In [ ]:
# Optional: run a larger scout set after initial sanity checks.
# full_specs = default_scout_model_grid()
# results_full, _ = run_quick_scout(
#     dataset=dataset,
#     scout_cfg=scout_cfg,
#     model_specs=full_specs,
#     seeds=SCOUT_SEEDS,
#     verbose=True,
# )
# results_full.head(20)


## Next Step

When scout metrics look stable, expand to a deeper search:

- widen `hidden_dims`
- tune `cross_layers` and `cross_rank`
- expand `margin_grid`
- run more seeds and walk-forward splits
